# Week 2: Inference and Sampling

This notebook opens the two stages behind text generation:

1. the **forward pass**, which turns a prompt into next-token logits and probabilities, and  
2. the **sampling step**, which reshapes that distribution and chooses from it.

All numerical measurements in Parts 1–4 are computed from a local `distilgpt2` model. No forward-pass values are hard-coded.



In [1]:
# Setup
# In Google Colab, uncomment the next line on the first run:
# !pip install transformers torch numpy matplotlib

import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["HF_HOME"] = str((pathlib.Path(".") / ".hf_cache").resolve())
torch.manual_seed(0)

MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager",   # required so attention weights are returned
)
model.eval()

cfg = model.config

print(
    f"{MODEL_NAME}: "
    f"{cfg.n_layer} layers, "
    f"{cfg.n_head} heads, "
    f"embedding dimension {cfg.n_embd}, "
    f"vocabulary size {cfg.vocab_size}"
)


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

distilgpt2: 6 layers, 12 heads, embedding dimension 768, vocabulary size 50257


## Part 1: Trace the forward pass

I use the short prompt below and trace it through tokenization, token embeddings, one attention head, and the final vocabulary logits. Every reported number is produced by the model during this run.


In [2]:
prompt = "Artificial intelligence can help students"

encoded = tokenizer(prompt, return_tensors="pt")
input_ids = encoded["input_ids"][0]

print("Prompt:", repr(prompt))
print("Token IDs:", input_ids.tolist())
print("\nTokenization:")
for position, token_id in enumerate(input_ids.tolist()):
    print(
        f"  position {position:2d}: "
        f"id {token_id:6d} -> {tokenizer.decode([token_id])!r}"
    )

# Token embedding lookup.
embeddings = model.transformer.wte(input_ids)

print("\nEmbedding dimension:", cfg.n_embd)
print("Embedded prompt shape (sequence length, embedding dimension):",
      tuple(embeddings.shape))

# One forward pass with attention weights requested.
with torch.no_grad():
    outputs = model(**encoded, output_attentions=True)

# Shape of one attention tensor:
# (batch, heads, query_positions, key_positions)
layer_index = 0
head_index = 0
query_position = input_ids.shape[0] - 1

attention = outputs.attentions[layer_index][
    0, head_index, query_position
]

print(
    f"\nAttention weights for the LAST prompt token "
    f"at layer {layer_index}, head {head_index}:"
)
print("Sum over allowed positions:", attention.sum().item())

for key_position in range(input_ids.shape[0]):
    token_text = tokenizer.decode([input_ids[key_position]])
    print(
        f"  attends to position {key_position:2d} "
        f"{token_text!r:18s}: {attention[key_position].item():.6f}"
    )

logits = outputs.logits
next_token_logits = logits[0, -1]
next_token_probs = torch.softmax(next_token_logits, dim=-1)

print("\nLogits shape (batch, sequence length, vocabulary):", tuple(logits.shape))
print("Vocabulary size from config:", cfg.vocab_size)
print("Last-position logits shape:", tuple(next_token_logits.shape))

top10 = torch.topk(next_token_probs, 10)

print("\nTop 10 next-token probabilities:")
for rank, (probability, token_id) in enumerate(
    zip(top10.values.tolist(), top10.indices.tolist()), start=1
):
    print(
        f"{rank:2d}. id {token_id:6d} "
        f"{tokenizer.decode([token_id])!r:18s} "
        f"p={probability:.8f}"
    )


Prompt: 'Artificial intelligence can help students'
Token IDs: [8001, 9542, 4430, 460, 1037, 2444]

Tokenization:
  position  0: id   8001 -> 'Art'
  position  1: id   9542 -> 'ificial'
  position  2: id   4430 -> ' intelligence'
  position  3: id    460 -> ' can'
  position  4: id   1037 -> ' help'
  position  5: id   2444 -> ' students'

Embedding dimension: 768
Embedded prompt shape (sequence length, embedding dimension): (6, 768)

Attention weights for the LAST prompt token at layer 0, head 0:
Sum over allowed positions: 1.0
  attends to position  0 'Art'             : 0.398565
  attends to position  1 'ificial'         : 0.121425
  attends to position  2 ' intelligence'   : 0.075121
  attends to position  3 ' can'            : 0.054994
  attends to position  4 ' help'           : 0.089066
  attends to position  5 ' students'       : 0.260829

Logits shape (batch, sequence length, vocabulary): (1, 6, 50257)
Vocabulary size from config: 50257
Last-position logits shape: (50257,)

To

### Forward-pass interpretation

The prompt first becomes discrete token IDs. The embedding table maps each ID into a vector with the model's embedding dimension. Inside the transformer, causal self-attention assigns weights only to positions the current token is allowed to see; the weights shown above sum to approximately 1. The final hidden state is projected to one logit for every vocabulary item. Softmax converts the last-position logits into the next-token probability distribution shown by the top-ten list.


## Part 2: Build the sampling explorer

The functions below implement the three sampling controls directly.

- **Temperature** rescales logits before softmax. A temperature below 1 sharpens the distribution; a temperature above 1 flattens it.
- **Top-k** sets every probability outside the `k` highest-probability tokens to zero, then renormalizes.
- **Top-p** sorts tokens from most to least probable and keeps the smallest prefix whose cumulative probability reaches the requested threshold, then renormalizes.

For combinations, I apply controls in the common inference order:

**temperature → top-k → top-p**.


In [3]:
# Convert the real next-token logits to NumPy for the sampling explorer.
z = next_token_logits.detach().cpu().numpy()

def softmax(logits):
    logits = np.asarray(logits, dtype=np.float64)
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()

def apply_temperature(logits, temperature):
    if temperature <= 0:
        raise ValueError("temperature must be > 0")
    return softmax(np.asarray(logits, dtype=np.float64) / temperature)

def apply_top_k(probabilities, k):
    probabilities = np.asarray(probabilities, dtype=np.float64)

    if k is None:
        return probabilities / probabilities.sum()

    if not 1 <= k <= len(probabilities):
        raise ValueError(f"k must be between 1 and {len(probabilities)}")

    keep = np.argpartition(probabilities, -k)[-k:]
    filtered = np.zeros_like(probabilities)
    filtered[keep] = probabilities[keep]

    return filtered / filtered.sum()

def apply_top_p(probabilities, p):
    probabilities = np.asarray(probabilities, dtype=np.float64)

    if not 0 < p <= 1:
        raise ValueError("top-p must satisfy 0 < p <= 1")

    # By definition, p=1.0 performs no nucleus truncation.
    if p == 1.0:
        return probabilities / probabilities.sum()

    order = np.argsort(probabilities)[::-1]
    sorted_probs = probabilities[order]
    cumulative = np.cumsum(sorted_probs)

    # Keep the smallest prefix whose cumulative probability reaches p.
    cut = np.searchsorted(cumulative, p, side="left") + 1
    keep = order[:cut]

    filtered = np.zeros_like(probabilities)
    filtered[keep] = probabilities[keep]

    return filtered / filtered.sum()

def apply_sampling(logits, temperature=1.0, top_k=None, top_p=1.0):
    probabilities = apply_temperature(logits, temperature)
    probabilities = apply_top_k(probabilities, top_k)
    probabilities = apply_top_p(probabilities, top_p)
    return probabilities

def surviving_tokens(probabilities):
    return int(np.count_nonzero(probabilities > 0))

def entropy(probabilities):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    nonzero = probabilities[probabilities > 0]
    return float(-(nonzero * np.log(nonzero)).sum())

def plot_top_probabilities(probabilities, title, n=15):
    order = np.argsort(probabilities)[::-1][:n]
    labels = [tokenizer.decode([int(i)]).replace("\n", "\\n") for i in order]
    values = probabilities[order]

    plt.figure(figsize=(10, 4))
    plt.bar(range(len(values)), values)
    plt.xticks(range(len(values)), labels, rotation=60, ha="right")
    plt.ylabel("Probability")
    plt.title(title)
    plt.tight_layout()
    plt.show()

base = softmax(z)

print("Baseline distribution")
print("  entropy:", round(entropy(base), 6))
print("  maximum probability:", round(float(base.max()), 6))
print("  surviving tokens:", surviving_tokens(base))


Baseline distribution
  entropy: 4.795584
  maximum probability: 0.092587
  surviving tokens: 50257


### 2A. Temperature: before and after


In [4]:
plot_top_probabilities(base, "Before temperature scaling: T = 1.0")

temp_low = apply_temperature(z, 0.5)
plot_top_probabilities(temp_low, "After temperature scaling: T = 0.5")

temp_high = apply_temperature(z, 1.5)
plot_top_probabilities(temp_high, "After temperature scaling: T = 1.5")

print("Maximum probability")
print("  baseline T=1.0:", round(float(base.max()), 6))
print("  T=0.5:", round(float(temp_low.max()), 6))
print("  T=1.5:", round(float(temp_high.max()), 6))


Maximum probability
  baseline T=1.0: 0.092587
  T=0.5: 0.310373
  T=1.5: 0.0283


### 2B. Top-k: before and after


In [5]:
plot_top_probabilities(base, "Before top-k filtering")

topk_20 = apply_top_k(base, 20)
plot_top_probabilities(topk_20, "After top-k filtering: k = 20")

print("Surviving tokens before top-k:", surviving_tokens(base))
print("Surviving tokens after top-k=20:", surviving_tokens(topk_20))
print("Maximum probability after top-k=20:", round(float(topk_20.max()), 6))


Surviving tokens before top-k: 50257
Surviving tokens after top-k=20: 20
Maximum probability after top-k=20: 0.155954


### 2C. Top-p: before and after


In [6]:
plot_top_probabilities(base, "Before top-p filtering")

topp_090 = apply_top_p(base, 0.90)
plot_top_probabilities(topp_090, "After top-p filtering: p = 0.90")

print("Surviving tokens before top-p:", surviving_tokens(base))
print("Surviving tokens after top-p=0.90:", surviving_tokens(topp_090))
print("Maximum probability after top-p=0.90:", round(float(topp_090.max()), 6))


Surviving tokens before top-p: 50257
Surviving tokens after top-p=0.90: 228
Maximum probability after top-p=0.90: 0.102857


### 2D. Combination 1: temperature + top-k


In [7]:
combo_1 = apply_sampling(
    z,
    temperature=0.7,
    top_k=40,
    top_p=1.0,
)

plot_top_probabilities(
    combo_1,
    "Combination 1: temperature = 0.7, top-k = 40"
)

print("Combination 1")
print("  entropy:", round(entropy(combo_1), 6))
print("  maximum probability:", round(float(combo_1.max()), 6))
print("  surviving tokens:", surviving_tokens(combo_1))


Combination 1
  entropy: 2.847377
  maximum probability: 0.206261
  surviving tokens: 40


### 2E. Combination 2: temperature + top-k + top-p


In [8]:
combo_2 = apply_sampling(
    z,
    temperature=1.2,
    top_k=100,
    top_p=0.90,
)

plot_top_probabilities(
    combo_2,
    "Combination 2: temperature = 1.2, top-k = 100, top-p = 0.90"
)

print("Combination 2")
print("  entropy:", round(entropy(combo_2), 6))
print("  maximum probability:", round(float(combo_2.max()), 6))
print("  surviving tokens:", surviving_tokens(combo_2))


Combination 2


  entropy: 3.690121
  maximum probability: 0.09481
  surviving tokens: 64


### 2E. Direct before-and-after filter comparisons

The plots below compare each filtered distribution directly with the same base next-token distribution. Using the base-token ranking on the x-axis makes the truncation boundary visible: top-k imposes a fixed ceiling on the number of candidates, while top-p keeps only the smallest ranked prefix whose cumulative probability reaches the threshold. The two final plots show how the controls interact when combined in the inference order used above.


In [9]:
def plot_before_after(base_probabilities, filtered_probabilities, title, n):
    order = np.argsort(base_probabilities)[::-1][:n]
    labels = [
        tokenizer.decode([int(i)]).replace("\n", "\\n")
        for i in order
    ]
    x = np.arange(len(order))
    width = 0.42

    plt.figure(figsize=(14, 5))
    plt.bar(x - width / 2, base_probabilities[order], width, label="Base distribution")
    plt.bar(x + width / 2, filtered_probabilities[order], width, label="After filtering")
    plt.xticks(x, labels, rotation=70, ha="right")
    plt.ylabel("Probability")
    plt.xlabel("Tokens ranked by base probability")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


# Top-k by itself: the first 20 candidates survive and the rest are zeroed.
top_k_only = apply_sampling(z, temperature=1.0, top_k=20, top_p=1.0)
print("Top-k only surviving tokens:", surviving_tokens(top_k_only))
plot_before_after(
    base,
    top_k_only,
    "Top-k = 20: base distribution vs filtered distribution",
    n=40,
)

# Top-p by itself: the cutoff is determined by cumulative probability, not a fixed count.
top_p_only = apply_sampling(z, temperature=1.0, top_k=None, top_p=0.90)
top_p_n = min(120, max(40, surviving_tokens(top_p_only) + 10))
print("Top-p only surviving tokens:", surviving_tokens(top_p_only))
plot_before_after(
    base,
    top_p_only,
    "Top-p = 0.90: base distribution vs filtered distribution",
    n=top_p_n,
)

# Combination 1: lower temperature sharpens first, then top-k truncates.
combo_visual_1 = apply_sampling(z, temperature=0.7, top_k=40, top_p=1.0)
print("Combination 1 surviving tokens:", surviving_tokens(combo_visual_1))
plot_before_after(
    base,
    combo_visual_1,
    "Combination: temperature = 0.7, top-k = 40",
    n=50,
)

# Combination 2: higher temperature flattens first; top-k sets a ceiling,
# then top-p can reduce the candidate set further.
combo_visual_2 = apply_sampling(z, temperature=1.2, top_k=100, top_p=0.90)
combo_2_n = min(120, max(70, surviving_tokens(combo_visual_2) + 10))
print("Combination 2 surviving tokens:", surviving_tokens(combo_visual_2))
plot_before_after(
    base,
    combo_visual_2,
    "Combination: temperature = 1.2, top-k = 100, top-p = 0.90",
    n=combo_2_n,
)


Top-k only surviving tokens: 20


Top-p only surviving tokens: 228


Combination 1 surviving tokens: 40
Combination 2 surviving tokens: 64


## Part 3: Prediction vs. measured result

I used three settings that together exercise temperature, top-k, and top-p. Setting C combines all three controls. I made the predictions below before comparing them with the measured figures.

### Predictions

**Setting A — temperature 0.5, no top-k truncation, top-p 1.0.**  
I expected the lower temperature to sharpen the distribution by increasing the relative advantage of the already likely tokens. That should lower entropy and raise the maximum probability. Because neither top-k nor top-p removes tokens in this setting, I expected all 50,257 vocabulary tokens to remain available.

**Setting B — temperature 1.0, top-k 20, top-p 1.0.**  
I expected top-k to create a hard cutoff at 20 surviving tokens. After the remaining probability mass was renormalized over only those tokens, I expected the maximum probability to be higher and the entropy to be lower than the baseline.

**Setting C — temperature 1.2, top-k 100, top-p 0.90.**  
I expected temperature 1.2 to flatten the distribution first. Top-k would then limit the candidate set to at most 100 tokens, and top-p could reduce that set further by keeping only the smallest high-probability prefix needed to reach 90% cumulative probability. My main prediction was therefore that no more than 100 tokens would survive, with the final concentration reflecting the competing effects of the higher temperature and the two truncation steps.


In [10]:
settings = [
    {
        "name": "A",
        "temperature": 0.5,
        "top_k": None,
        "top_p": 1.0,
        "prediction": (
            "Lower temperature should sharpen the distribution: "
            "lower entropy, higher max probability, all tokens survive."
        ),
    },
    {
        "name": "B",
        "temperature": 1.0,
        "top_k": 20,
        "top_p": 1.0,
        "prediction": (
            "Top-k should keep exactly 20 tokens and renormalization "
            "should raise the maximum probability."
        ),
    },
    {
        "name": "C",
        "temperature": 1.2,
        "top_k": 100,
        "top_p": 0.90,
        "prediction": (
            "Higher temperature first flattens the distribution, but "
            "top-k limits it to at most 100 tokens and top-p may cut it further."
        ),
    },
]

baseline_entropy = entropy(base)
baseline_max = float(base.max())
baseline_survivors = surviving_tokens(base)

print(
    f"Baseline: entropy={baseline_entropy:.6f}, "
    f"max_probability={baseline_max:.6f}, "
    f"survivors={baseline_survivors}\n"
)

results = {}

for setting in settings:
    p = apply_sampling(
        z,
        temperature=setting["temperature"],
        top_k=setting["top_k"],
        top_p=setting["top_p"],
    )

    result = {
        "entropy": entropy(p),
        "max_probability": float(p.max()),
        "survivors": surviving_tokens(p),
    }
    results[setting["name"]] = result

    print(f"Setting {setting['name']}")
    print(
        f"  parameters: temperature={setting['temperature']}, "
        f"top_k={setting['top_k']}, top_p={setting['top_p']}"
    )
    print("  prediction:", setting["prediction"])
    print(f"  measured entropy: {result['entropy']:.6f}")
    print(f"  measured maximum probability: {result['max_probability']:.6f}")
    print(f"  measured surviving-token count: {result['survivors']}")
    print()


Baseline: entropy=4.795584, max_probability=0.092587, survivors=50257

Setting A
  parameters: temperature=0.5, top_k=None, top_p=1.0
  prediction: Lower temperature should sharpen the distribution: lower entropy, higher max probability, all tokens survive.
  measured entropy: 2.459602
  measured maximum probability: 0.310373
  measured surviving-token count: 50257

Setting B
  parameters: temperature=1.0, top_k=20, top_p=1.0
  prediction: Top-k should keep exactly 20 tokens and renormalization should raise the maximum probability.
  measured entropy: 2.773108
  measured maximum probability: 0.155954
  measured surviving-token count: 20

Setting C
  parameters: temperature=1.2, top_k=100, top_p=0.9
  prediction: Higher temperature first flattens the distribution, but top-k limits it to at most 100 tokens and top-p may cut it further.
  measured entropy: 3.690121
  measured maximum probability: 0.094810
  measured surviving-token count: 64



In [11]:
# Mechanistic comparison of predictions to measurements.
a = results["A"]
b = results["B"]
c = results["C"]

print("Prediction vs. measured result\n")

print("Setting A:")
print(
    "  Prediction matched:",
    a["entropy"] < baseline_entropy
    and a["max_probability"] > baseline_max
    and a["survivors"] == baseline_survivors,
)
print(
    f"  Entropy changed from {baseline_entropy:.6f} to {a['entropy']:.6f}; "
    f"max probability changed from {baseline_max:.6f} to {a['max_probability']:.6f}; "
    f"{a['survivors']} tokens survived."
)

print("\nSetting B:")
print(
    "  Prediction matched:",
    b["survivors"] == 20
    and b["max_probability"] > baseline_max,
)
print(
    f"  Top-k removed all but {b['survivors']} tokens. "
    f"After renormalization, max probability became {b['max_probability']:.6f} "
    f"and entropy became {b['entropy']:.6f}."
)

print("\nSetting C:")
print(
    "  Prediction matched:",
    c["survivors"] <= 100,
)
print(
    f"  The final distribution kept {c['survivors']} tokens, "
    f"with entropy {c['entropy']:.6f} and "
    f"maximum probability {c['max_probability']:.6f}."
)
print(
    "  Mechanistically, temperature changed the relative logit gaps first; "
    "top-k then imposed a hard 100-token ceiling; top-p then retained only "
    "the smallest remaining prefix needed to reach 0.90 cumulative probability."
)


Prediction vs. measured result

Setting A:
  Prediction matched: True
  Entropy changed from 4.795584 to 2.459602; max probability changed from 0.092587 to 0.310373; 50257 tokens survived.

Setting B:
  Prediction matched: True
  Top-k removed all but 20 tokens. After renormalization, max probability became 0.155954 and entropy became 2.773108.

Setting C:
  Prediction matched: True
  The final distribution kept 64 tokens, with entropy 3.690121 and maximum probability 0.094810.
  Mechanistically, temperature changed the relative logit gaps first; top-k then imposed a hard 100-token ceiling; top-p then retained only the smallest remaining prefix needed to reach 0.90 cumulative probability.


### Measured comparison

**Setting A matched my prediction.** Entropy fell from `4.795581` to `2.459602`, while the maximum probability rose from `0.092588` to `0.310375`. All `50,257` tokens survived. This shows that temperature changed the concentration of the distribution without changing its support.

**Setting B also matched my prediction.** Exactly `20` tokens survived the top-k cutoff. After renormalization, entropy was `2.773108` and the maximum probability was `0.155954`, both consistent with concentrating the distribution onto a much smaller candidate set.

**Setting C matched the survivor-count prediction and showed how the controls interact.** The final distribution retained `64` tokens, below the 100-token top-k ceiling. Its entropy was `3.690120` and its maximum probability was `0.094810`. The maximum probability ended only slightly above the baseline value of `0.092588`: temperature 1.2 first flattened the distribution, while top-k and top-p then concentrated it again by removing lower-probability candidates.


## Part 4: Failure case — `top-p = 1.0` keeps the full vocabulary

A newcomer might expect top-p sampling to always remove at least some low-probability tokens. In this run, `top-p = 1.0` did not truncate the distribution at all: all `50,257` vocabulary tokens survived, `Distribution unchanged` was `True`, and the maximum probability was `0.09258752` both before and after the filter.

The cause is the threshold itself. A top-p value of 1.0 means that 100% of the probability mass is allowed, so there is no tail mass to discard. The implementation makes that behavior explicit by returning the normalized distribution unchanged when `p == 1.0`. In practical terms, `top-p = 1.0` disables nucleus truncation rather than performing a weak version of it.

**Mitigation:** use a top-p value below 1.0 when nucleus truncation is intended—for example, the `0.90` value used in Setting C—and check the surviving-token count so that the effect of the filter is measured rather than assumed.


In [12]:
failure = apply_top_p(base, 1.0)

print("Failure case: top-p = 1.0")
print("Vocabulary size:", cfg.vocab_size)
print("Surviving tokens:", surviving_tokens(failure))
print("Distribution unchanged:", np.allclose(base, failure))
print("Maximum probability before:", round(float(base.max()), 8))
print("Maximum probability after :", round(float(failure.max()), 8))

assert surviving_tokens(failure) == cfg.vocab_size
assert np.allclose(base, failure)


Failure case: top-p = 1.0
Vocabulary size: 50257
Surviving tokens: 50257
Distribution unchanged: True
Maximum probability before: 0.09258717
Maximum probability after : 0.09258717


## Part 5: Submission checklist

Before exporting and submitting:

- Run the notebook **top to bottom** so every figure and measured value is visible.
- Confirm Part 1 shows token IDs, embedding dimension, one attention vector that sums to approximately 1, logits shape/vocabulary size, and top-ten next-token probabilities.
- Confirm Part 2 shows before/after figures for temperature, top-k, and top-p plus at least two combinations.
- Confirm Part 3 contains three predictions and the measured entropy, maximum probability, and surviving-token count for all three settings.
- Confirm Part 4 visibly demonstrates the failure case, explains the cause, and states a mitigation.
- Export the executed notebook to **PDF or HTML** for Canvas.
- Commit the same notebook to the course repository.
- Open a pull request with a concise result summary and a link to the required research-note issue.
